# Assignment 2: Dynamic Chunking and a FAISS Vector Database

**Course:** Developing Generative AI Solutions

**Student:** write your name here

This notebook builds a retrieval pipeline over three text files from different fields and stores the vectors in a FAISS vector database on disk.

**Phase 1 (runs once):**

1. Load three text files: Saudi cities, the solar system, and the human body.
2. Split them with dynamic chunking: a paragraph becomes one chunk, and a paragraph that is too long is split at sentence ends.
3. Embed every chunk with `gemini-embedding-001`.
4. Store the vectors in a FAISS index and save it to disk, together with the chunk texts and their file names.

**Phase 2 (runs every time):**

1. Load the saved vector database from disk.
2. Take a search query from the user and embed it with the same model.
3. Search the FAISS index with cosine similarity.
4. Print the question and the top 3 chunks, each with its score and its reference (file name).

If the vector database already exists on disk, Phase 1 is skipped. A new run does not chunk or embed the documents again.

**To run:** add a Gemini API key to Colab Secrets under the name `GEMINI_API_KEY` and turn on notebook access for it. Then run the cells from top to bottom.

## Setup

Install the Gemini SDK and FAISS, import the libraries, create the API client, and define the settings.

In [ ]:
!pip install -q google-genai faiss-cpu

In [ ]:
import os
import re
import json
import time
import numpy as np
import faiss
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

EMBED_MODEL = "gemini-embedding-001"
DATA_FILES = ["saudi_cities.txt", "solar_system.txt", "human_body.txt"]
MAX_CHUNK_CHARS = 600
TOP_K = 3

DB_DIR = "vector_db"
INDEX_PATH = os.path.join(DB_DIR, "index.faiss")   # the vectors
META_PATH = os.path.join(DB_DIR, "chunks.json")    # the chunk texts and file names

## Data files

These three cells create the text files inside Colab. Each file covers a different field: geography, astronomy, and health.

In [ ]:
%%writefile saudi_cities.txt
الرياض عاصمة المملكة العربية السعودية وأكبر مدنها من حيث عدد السكان، وتقع في وسط المملكة على هضبة نجد. من أشهر معالمها التاريخية قصر المصمك الذي استعاده الملك عبدالعزيز عام 1902، وبالقرب منها تقع الدرعية عاصمة الدولة السعودية الأولى، وفيها حي الطريف المسجل في قائمة التراث العالمي لليونسكو منذ عام 2010. ومن معالمها الحديثة برج المملكة وبرج الفيصلية ومركز الملك عبدالله المالي. افتُتح قطار الرياض في أواخر عام 2024، وهو شبكة مترو من ستة مسارات. وقد فازت الرياض باستضافة معرض إكسبو 2030.

مكة المكرمة أقدس مدينة في الإسلام، وتقع في غرب المملكة. فيها المسجد الحرام الذي تتوسطه الكعبة المشرفة، قبلة المسلمين في صلاتهم، ويقصدها ملايين المسلمين كل عام لأداء الحج والعمرة. وُلد فيها النبي محمد صلى الله عليه وسلم، وفيها جبل النور الذي يضم غار حراء حيث نزل الوحي أول مرة. بالقرب منها المشاعر المقدسة: منى وعرفات ومزدلفة. ومن أبرز مبانيها الحديثة برج الساعة المطل على المسجد الحرام، وهو من أطول المباني في العالم.

المدينة المنورة ثاني أقدس مدينة في الإسلام، وتقع في غرب المملكة شمال مكة المكرمة. كانت تُسمى يثرب قبل أن يهاجر إليها النبي محمد صلى الله عليه وسلم عام 622 للميلاد، وهو الحدث الذي يبدأ به التقويم الهجري. فيها المسجد النبوي الذي يضم قبر النبي، ومسجد قباء وهو أول مسجد بُني في الإسلام. ويقع شمالها جبل أحد الذي وقعت عنده غزوة أحد في السنة الثالثة للهجرة. تشتهر المدينة بالتمور وخاصة تمر العجوة، وفيها مجمع الملك فهد لطباعة المصحف الشريف.

جدة مدينة ساحلية على البحر الأحمر، وتُعرف ببوابة الحرمين الشريفين لأن كثيرًا من الحجاج والمعتمرين يصلون عبر مطارها ومينائها. فيها ميناء جدة الإسلامي، وهو الميناء الرئيسي للمملكة على البحر الأحمر، ومطار الملك عبدالعزيز الدولي. سُجّلت منطقة جدة التاريخية، المعروفة بالبلد، في قائمة التراث العالمي لليونسكو عام 2014، وتتميز ببيوتها القديمة ذات الرواشين الخشبية. ومن معالمها نافورة الملك فهد، وهي أعلى نافورة في العالم. وتستضيف جدة سباق الفورمولا 1 على حلبة الكورنيش منذ عام 2021.

الدمام عاصمة المنطقة الشرقية، وتقع على ساحل الخليج العربي، وتشكّل مع الخبر والظهران تجمعًا حضريًا واحدًا. ارتبط اسمها بالنفط، ففي عام 1938 اكتُشف النفط بكميات تجارية في بئر الدمام رقم 7 المعروفة ببئر الخير، وتتخذ شركة أرامكو السعودية من الظهران مقرًّا لها. في الدمام ميناء الملك عبدالعزيز، وهو من أكبر موانئ المملكة على الخليج العربي، ويخدمها مطار الملك فهد الدولي، وهو أكبر مطار في العالم من حيث المساحة. ويربط جسر الملك فهد، الذي افتُتح عام 1986، مدينة الخبر بمملكة البحرين.

الطائف مدينة جبلية في منطقة مكة المكرمة، تقع على مرتفعات جبال السروات على ارتفاع يتجاوز 1700 متر فوق سطح البحر. يجعلها مناخها المعتدل مصيفًا يقصده الناس هربًا من حرارة الصيف، ولذلك تُلقَّب بعروس المصايف. تشتهر بالورد الطائفي الذي يُستخرج منه دهن الورد وماء الورد، وبفواكهها مثل الرمان والعنب. ومن أبرز مواقعها منطقتا الهدا والشفا الجبليتان، وسوق عكاظ الذي كان أشهر أسواق العرب قبل الإسلام وكان الشعراء يتبارون فيه.

أبها عاصمة منطقة عسير في جنوب غرب المملكة، وتقع على ارتفاع نحو 2200 متر فوق سطح البحر، ولذلك يبقى طقسها معتدلًا في الصيف وتكثر فيها الأمطار والضباب. بالقرب منها جبل السودة، وهو من أعلى القمم في المملكة ويبلغ ارتفاعه نحو ثلاثة آلاف متر. ومن المواقع التراثية القريبة منها قرية رجال ألمع بمبانيها الحجرية العالية. وتشتهر المنطقة بفن القط العسيري، وهو زخارف ملونة ترسمها النساء على جدران البيوت، وقد سُجّل في قائمة التراث الثقافي غير المادي لليونسكو عام 2017.

تبوك عاصمة منطقة تبوك في شمال غرب المملكة. عُرفت في التاريخ الإسلامي بغزوة تبوك التي وقعت في السنة التاسعة للهجرة، ومن معالمها التاريخية قلعة تبوك ومحطة سكة حديد الحجاز. تعتمد المنطقة على الزراعة وتشتهر بإنتاج الورود والفواكه والقمح. في منطقة تبوك جبل اللوز الذي تتساقط عليه الثلوج في بعض فصول الشتاء، وفيها كذلك مشروع نيوم على ساحل البحر الأحمر، وهو من أكبر المشاريع في رؤية المملكة 2030.

حائل عاصمة منطقة حائل في شمال المملكة، وتقع بين سلسلتي جبال أجا وسلمى. اشتهرت بأنها موطن حاتم الطائي الذي يُضرب به المثل في الكرم عند العرب. من معالمها قلعة أعيرف المبنية على تل يطل على المدينة. وفي منطقة حائل موقعا جبة والشويمس اللذان يضمان نقوشًا ورسومًا صخرية عمرها آلاف السنين، وقد سُجّلا في قائمة التراث العالمي لليونسكو عام 2015. وتستضيف حائل كل عام رالي حائل الدولي في صحراء النفود الكبير.

بريدة عاصمة منطقة القصيم في وسط المملكة، وهي من أهم المدن الزراعية فيها. تشتهر بإنتاج التمور، وخاصة التمر السكري، ويقام فيها كل عام مهرجان للتمور يُعد من أكبر أسواق التمور في العالم. وفيها كذلك سوق للإبل من أكبر أسواق الإبل في العالم. تعتمد الزراعة في القصيم على المياه الجوفية، وتنتج المنطقة إلى جانب التمور القمح والخضراوات. وتقع بالقرب منها مدينة عنيزة، ثاني أكبر مدن القصيم.

الأحساء محافظة في المنطقة الشرقية، وأكبر مدنها الهفوف والمبرز. تضم واحة الأحساء، وهي أكبر واحة نخيل في العالم وفيها نحو مليونين ونصف المليون نخلة، وقد سُجّلت في قائمة التراث العالمي لليونسكو عام 2018. تشتهر الأحساء بتمر الخلاص وبعيون الماء الطبيعية. ومن معالمها جبل القارة المعروف بكهوفه الباردة صيفًا، وقصر إبراهيم التاريخي في الهفوف، ومسجد جواثا وهو من أقدم المساجد في الإسلام.

العلا محافظة في منطقة المدينة المنورة في شمال غرب المملكة، وتشتهر بآثارها وتكويناتها الصخرية. فيها موقع الحِجر، المعروف بمدائن صالح، وهو أول موقع سعودي يُسجَّل في قائمة التراث العالمي لليونسكو، وكان ذلك عام 2008. يضم الحِجر أكثر من مئة مقبرة نحتها الأنباط في الصخر قبل نحو ألفي عام. ومن معالم العلا جبل الفيل، وهو صخرة طبيعية تشبه شكل الفيل، والبلدة القديمة، وقاعة مرايا وهي أكبر مبنى مغطى بالمرايا في العالم.

In [ ]:
%%writefile solar_system.txt
الشمس نجم يقع في مركز المجموعة الشمسية، وتدور حولها الكواكب الثمانية وما يتبعها من أقمار وكويكبات ومذنبات. تشكّل الشمس نحو 99.8% من كتلة المجموعة الشمسية كلها، وتتكون أساسًا من الهيدروجين والهيليوم. تبلغ درجة حرارة سطحها نحو 5500 درجة مئوية، ويحتاج ضوؤها إلى نحو ثماني دقائق ليصل إلى الأرض. تنتج الشمس طاقتها من الاندماج النووي في باطنها، حيث يتحول الهيدروجين إلى هيليوم.

عطارد أقرب الكواكب إلى الشمس وأصغرها حجمًا. يُتم دورته حول الشمس في 88 يومًا أرضيًا، وهي أقصر سنة بين كواكب المجموعة الشمسية. ليس لعطارد أقمار، وغلافه الجوي رقيق جدًا، ولذلك يكون الفرق في درجات الحرارة بين نهاره وليله كبيرًا جدًا. يشبه سطحه سطح القمر، فهو مليء بالفوهات التي خلّفتها اصطدامات النيازك.

الزهرة ثاني الكواكب بعدًا عن الشمس، وهو أشد كواكب المجموعة الشمسية حرارة، إذ تبلغ درجة حرارة سطحه نحو 465 درجة مئوية. سبب هذه الحرارة غلافه الجوي الكثيف المكوَّن في معظمه من ثاني أكسيد الكربون، وهو يحبس الحرارة. يدور كوكب الزهرة حول نفسه في اتجاه معاكس لاتجاه دوران معظم الكواكب، ويومه أطول من سنته: يدور حول نفسه في 243 يومًا أرضيًا، ويدور حول الشمس في 225 يومًا. وليس للزهرة أقمار.

الأرض ثالث الكواكب بعدًا عن الشمس، وهي الكوكب الوحيد المعروف بوجود الحياة عليه. يغطي الماء نحو 71% من سطحها. تُتم الأرض دورتها حول الشمس في 365 يومًا وربع اليوم، وتدور حول نفسها مرة كل 24 ساعة. يميل محور دورانها بنحو 23.5 درجة، وهذا الميل هو سبب تعاقب الفصول الأربعة. يحمي الغلاف الجوي والمجال المغناطيسي الأرضَ من الإشعاعات الضارة القادمة من الفضاء.

القمر هو التابع الطبيعي الوحيد للأرض، ويبعد عنها نحو 384 ألف كيلومتر. يُتم دورته حول الأرض في نحو 27 يومًا. جاذبية القمر هي السبب الرئيسي في ظاهرة المد والجزر في البحار والمحيطات. هبط الإنسان على سطح القمر أول مرة عام 1969 في رحلة أبولو 11، وكان رائد الفضاء نيل أرمسترونغ أول من مشى عليه.

المريخ رابع الكواكب بعدًا عن الشمس، ويُعرف بالكوكب الأحمر بسبب أكسيد الحديد الذي يغطي سطحه. فيه جبل أوليمبوس، وهو أعلى بركان معروف في المجموعة الشمسية ويبلغ ارتفاعه نحو 22 كيلومترًا. للمريخ قمران صغيران هما فوبوس وديموس. يومه قريب من يوم الأرض، فطوله 24 ساعة و37 دقيقة، أما سنته فتعادل 687 يومًا أرضيًا. وقد أرسلت وكالات الفضاء عدة مركبات جوّالة لدراسة سطحه.

المشتري خامس الكواكب بعدًا عن الشمس وأكبر كواكب المجموعة الشمسية، وهو عملاق غازي يتكون في معظمه من الهيدروجين والهيليوم. من أشهر ما يميّزه البقعة الحمراء العظيمة، وهي عاصفة هائلة مستمرة منذ مئات السنين. للمشتري أكثر من تسعين قمرًا معروفًا، أكبرها غانيميد، وهو أكبر قمر في المجموعة الشمسية. يدور المشتري حول نفسه بسرعة كبيرة، فيومه أقل من عشر ساعات.

زحل سادس الكواكب بعدًا عن الشمس وثاني أكبرها حجمًا، ويشتهر بحلقاته الواسعة المكوَّنة من قطع الجليد والصخور. هو أقل الكواكب كثافة، فكثافته أقل من كثافة الماء. لزحل عدد كبير من الأقمار، أشهرها تيتان، وهو القمر الوحيد في المجموعة الشمسية الذي له غلاف جوي كثيف.

أورانوس سابع الكواكب بعدًا عن الشمس، وهو عملاق جليدي يميل لونه إلى الأزرق المخضر بسبب غاز الميثان في غلافه الجوي. يميل محور دورانه بنحو 98 درجة، فيبدو كأنه يدور على جانبه. اكتشفه الفلكي ويليام هيرشل عام 1781، وكان أول كوكب يُكتشف باستخدام التلسكوب.

نبتون ثامن الكواكب وأبعدها عن الشمس. يحتاج إلى نحو 165 سنة أرضية ليُتم دورة واحدة حول الشمس. تهب عليه أقوى رياح مسجَّلة في المجموعة الشمسية. اكتُشف نبتون عام 1846 بعد أن تنبأ العلماء بموقعه بالحسابات الرياضية قبل رصده بالتلسكوب. أكبر أقماره هو ترايتون.

بلوتو كان يُعد الكوكب التاسع حتى عام 2006، حين أعاد الاتحاد الفلكي الدولي تصنيفه كوكبًا قزمًا. يقع بلوتو في حزام كايبر، وهو منطقة من الأجرام الجليدية وراء مدار نبتون. زارته المركبة نيو هورايزنز عام 2015 وأرسلت أول صور قريبة لسطحه. وبين مداري المريخ والمشتري يقع حزام الكويكبات، وأكبر أجرامه سيريس الذي يُصنَّف كوكبًا قزمًا أيضًا.

In [ ]:
%%writefile human_body.txt
القلب عضلة بحجم قبضة اليد تقريبًا، وتقع في الصدر بين الرئتين. يتكون من أربع حجرات: أذينين وبطينين. يضخ القلب الدم إلى جميع أنحاء الجسم عبر الأوعية الدموية، وينبض نحو مئة ألف مرة في اليوم. يتراوح معدل نبض القلب الطبيعي عند البالغ وقت الراحة بين 60 و100 نبضة في الدقيقة.

يتكون الدم من البلازما وخلايا الدم الحمراء وخلايا الدم البيضاء والصفائح الدموية. تنقل خلايا الدم الحمراء الأكسجين إلى أنسجة الجسم بواسطة بروتين الهيموغلوبين، وتدافع خلايا الدم البيضاء عن الجسم ضد العدوى، وتساعد الصفائح الدموية على تخثر الدم عند الجروح. في جسم الإنسان البالغ نحو خمسة لترات من الدم. وتنقسم فصائل الدم الرئيسية إلى أربع: A و B و AB و O.

الرئتان هما العضوان الرئيسيان في الجهاز التنفسي. يدخل الهواء عبر القصبة الهوائية ثم يتوزع في الشعب الهوائية حتى يصل إلى الحويصلات الهوائية، وفيها ينتقل الأكسجين إلى الدم ويخرج ثاني أكسيد الكربون. تتكون الرئة اليمنى من ثلاثة فصوص، وتتكون اليسرى من فصين. والحجاب الحاجز هو العضلة الرئيسية المسؤولة عن حركة التنفس.

الدماغ مركز التحكم في الجسم، ويزن عند البالغ نحو 1.4 كيلوغرام. يتكون من ثلاثة أجزاء رئيسية: المخ وهو المسؤول عن التفكير والذاكرة والحواس، والمخيخ وهو المسؤول عن التوازن وتناسق الحركة، وجذع الدماغ الذي ينظم الوظائف الحيوية مثل التنفس ونبض القلب. يحتوي الدماغ على نحو 86 مليار خلية عصبية، ويستهلك نحو 20% من طاقة الجسم.

يتكون الهيكل العظمي للإنسان البالغ من 206 عظام. أطول عظام الجسم عظم الفخذ، وأصغرها عظم الركاب في الأذن الوسطى. تدعم العظام الجسم وتحمي الأعضاء الداخلية، فالجمجمة تحمي الدماغ والقفص الصدري يحمي القلب والرئتين. تخزّن العظام الكالسيوم، ويُنتج نخاع العظم خلايا الدم.

يبدأ الجهاز الهضمي بالفم، ثم المريء، ثم المعدة التي تفرز العصارات الحمضية لهضم الطعام. يُمتص معظم الغذاء في الأمعاء الدقيقة التي يبلغ طولها نحو ستة أمتار، ويُمتص الماء في الأمعاء الغليظة. والكبد أكبر عضو داخلي في الجسم، ويفرز العصارة الصفراوية التي تساعد على هضم الدهون.

الجلد أكبر أعضاء جسم الإنسان، ويتكون من ثلاث طبقات: البشرة والأدمة والطبقة تحت الجلد. يحمي الجلد الجسم من الجراثيم ومن فقدان الماء، وينظم درجة حرارته عن طريق التعرّق. ويصنع الجلد فيتامين د عند تعرضه لأشعة الشمس.

الكليتان عضوان يقعان في أسفل الظهر على جانبي العمود الفقري. تنقّي الكليتان الدم من الفضلات والسوائل الزائدة وتكوّنان البول، وتنظمان توازن الماء والأملاح في الجسم. تحتوي كل كلية على نحو مليون وحدة ترشيح صغيرة تسمى النفرونات.

النوم ضروري لصحة الجسم والعقل. يحتاج البالغ إلى ما بين سبع وتسع ساعات من النوم كل ليلة. يمر النوم بمراحل متعاقبة، منها مرحلة حركة العين السريعة التي تكثر فيها الأحلام. تؤثر قلة النوم في التركيز والذاكرة، وتضعف جهاز المناعة.

يشكّل الماء نحو 60% من وزن جسم الإنسان البالغ. يحتاج الجسم إلى ثلاثة عناصر غذائية كبرى: الكربوهيدرات وهي المصدر الرئيسي للطاقة، والبروتينات لبناء العضلات والأنسجة، والدهون. ويحتاج كذلك إلى الفيتامينات والمعادن: فيتامين ج موجود في الحمضيات، والحديد ضروري لتكوين الهيموغلوبين، والكالسيوم ضروري لصحة العظام والأسنان.

النشاط البدني المنتظم يقوّي القلب والعضلات والعظام. توصي منظمة الصحة العالمية البالغين بممارسة نشاط بدني معتدل الشدة لمدة 150 دقيقة على الأقل في الأسبوع. يساعد النشاط البدني على ضبط الوزن وتحسين المزاج وجودة النوم، ويقلل خطر الإصابة بأمراض القلب والسكري من النوع الثاني.

## Embedding functions

`embed` sends texts to the embedding model in batches and waits and retries if the API is busy. `normalize` scales every vector to length 1. With vectors of length 1, the inner product that FAISS computes is equal to cosine similarity.

In [ ]:
def embed(texts, task_type, batch_size=50):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        for attempt in range(1, 6):
            try:
                result = client.models.embed_content(
                    model=EMBED_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(task_type=task_type),
                )
                break
            except Exception as error:
                code = getattr(error, "code", None)
                if code not in (429, 500, 503) or attempt == 5:
                    raise
                print(f"The API is busy (code {code}). Waiting {20 * attempt} seconds...")
                time.sleep(20 * attempt)
        if len(result.embeddings) != len(batch):
            raise RuntimeError(f"Sent {len(batch)} texts but received {len(result.embeddings)} vectors.")
        vectors.extend(e.values for e in result.embeddings)
    return np.array(vectors, dtype="float32")

def normalize(vectors):
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

## Dynamic chunking

The text is split at its natural boundaries instead of at a fixed character count. Each paragraph that fits within `MAX_CHUNK_CHARS` becomes one chunk. A longer paragraph is split into sentences, and the sentences are grouped into chunks that stay under the limit. Chunk sizes vary, and every chunk starts and ends at a sentence boundary.

In [ ]:
def split_paragraphs(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?؟])\s+", text) if s.strip()]

def dynamic_chunks(text, max_chars=MAX_CHUNK_CHARS):
    chunks = []
    for paragraph in split_paragraphs(text):
        if len(paragraph) <= max_chars:
            chunks.append(paragraph)
            continue
        current = ""
        for sentence in split_sentences(paragraph):
            if current and len(current) + 1 + len(sentence) > max_chars:
                chunks.append(current)
                current = sentence
            else:
                current = (current + " " + sentence).strip()
        if current:
            chunks.append(current)
    return chunks

## Phase 1: Build the vector database (runs once)

`build_vector_db` loads the files, chunks them, embeds the chunks, adds the vectors to a FAISS index, and saves two files: `index.faiss` with the vectors and `chunks.json` with each chunk's text and source file name.

Before building, the code checks whether both files already exist. If they do, Phase 1 is skipped. To force a rebuild, delete the `vector_db` folder.

In [ ]:
def build_vector_db():
    chunks = []
    for name in DATA_FILES:
        with open(name, "r", encoding="utf-8") as f:
            text = f.read()
        pieces = dynamic_chunks(text)
        for piece in pieces:
            chunks.append({"id": len(chunks), "source": name, "text": piece})
        print(f"{name}: {len(text)} characters -> {len(pieces)} chunks")

    vectors = normalize(embed([c["text"] for c in chunks], "RETRIEVAL_DOCUMENT"))
    print("Vectors shape:", vectors.shape)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)

    os.makedirs(DB_DIR, exist_ok=True)
    faiss.write_index(index, INDEX_PATH)
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump({"embed_model": EMBED_MODEL, "chunks": chunks}, f, ensure_ascii=False, indent=2)
    print(f"Saved {index.ntotal} vectors to {DB_DIR}/")


if os.path.exists(INDEX_PATH) and os.path.exists(META_PATH):
    print("Vector DB found on disk. Phase 1 skipped.")
else:
    print("Vector DB not found. Running Phase 1...")
    build_vector_db()

## Phase 2: Search

### Load the vector database from disk

Phase 2 reads everything it needs from the saved files, so it works in a run where Phase 1 was skipped. It also checks that the database was built with the same embedding model that will embed the query.

In [ ]:
index = faiss.read_index(INDEX_PATH)
with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)
chunks = meta["chunks"]

if meta["embed_model"] != EMBED_MODEL:
    raise RuntimeError("The saved DB was built with a different embedding model. Delete the vector_db folder and run again.")

print("Vectors in the index:", index.ntotal)
print("Chunks loaded:", len(chunks))

### Get the user's query

The query is typed by the user while the program runs. Examples: ما المدينة التي تشتهر بالورد؟ or ما أكبر كواكب المجموعة الشمسية؟ or كم عدد عظام جسم الإنسان البالغ؟

In [ ]:
question = input("Enter your question: ").strip()

### Embed the query, search, and print the results

The query is embedded with the same model and normalized. `index.search` compares it with every stored vector and returns the 3 highest cosine similarity scores with the ids of their chunks. Each result is printed with its score and its reference, the name of the file it came from.

In [ ]:
query_vector = normalize(embed([question], "RETRIEVAL_QUERY"))
scores, ids = index.search(query_vector, TOP_K)

print("Question:", question)
print()
for rank, (score, i) in enumerate(zip(scores[0], ids[0]), start=1):
    chunk = chunks[int(i)]
    print(f"Answer {rank} | score = {score:.4f} | reference: {chunk['source']}")
    print(chunk["text"])
    print("-" * 70)